# Card Transaction Bronze Generator — JSON with Data Quality Errors

This notebook generates synthetic bronze-layer card transaction data as **JSON files** in a UC Volume (`workshop_guy_catalog.fe_bar_a0.staging`). Each entity writes to its own subdirectory: `customers/`, `merchants/`, `transactions/`, and `prior_fraud_cases/`. Approximately **15%** of rows are injected with data quality errors (duplicate rows, null `txn_id`, null `customer_id`) to simulate realistic raw-landing data for downstream cleansing pipelines.

In [0]:
%uv pip install Faker

In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import DoubleType, StringType
from pyspark.sql.functions import pandas_udf
import pandas as pd

SEED = 42
CATALOG = "workshop_guy_catalog"
SCHEMA = "fe_bar_a0"
VOLUME = "staging"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
CUSTOMERS_PATH = f"{VOLUME_ROOT}/customers"
MERCHANTS_PATH = f"{VOLUME_ROOT}/merchants"
TRANSACTIONS_PATH = f"{VOLUME_ROOT}/transactions"
CASES_PATH = f"{VOLUME_ROOT}/prior_fraud_cases"

# Data quality error injection rate
DQ_ERROR_RATE = 0.15  # 15% of rows will have quality issues
DQ_DUPLICATE_SHARE = 0.50  # half of errors are duplicates
DQ_NULL_TXN_ID_SHARE = 0.25  # quarter are null txn_id
DQ_NULL_CUST_ID_SHARE = 0.25  # quarter are null customer_id

N_CUSTOMERS = 10_000
N_MERCHANTS = 5_000
N_TRANSACTIONS = 500_000
N_CASES = 50_000
DIM_PARTITIONS = 16
FACT_PARTITIONS = 32

HOME_REGION_WEIGHTS = [
    ("US-East", 0.30),
    ("US-West", 0.25),
    ("EU", 0.20),
    ("APAC", 0.15),
    ("LATAM", 0.10),
]
RISK_SEGMENT_WEIGHTS = [
    ("low", 0.60),
    ("medium", 0.25),
    ("high", 0.15),
]
MCC_WEIGHTS = [
    ("retail", 0.30),
    ("dining", 0.20),
    ("travel", 0.15),
    ("online_services", 0.15),
    ("groceries", 0.10),
    ("entertainment", 0.10),
]
MERCHANT_RISK_WEIGHTS = [
    ("low", 0.50),
    ("medium", 0.30),
    ("high", 0.20),
]
COUNTRY_WEIGHTS = [
    ("US", 0.40),
    ("UK", 0.15),
    ("DE", 0.10),
    ("JP", 0.08),
    ("BR", 0.07),
    ("IN", 0.05),
    ("CA", 0.05),
    ("MX", 0.04),
    ("FR", 0.03),
    ("SG", 0.03),
]
CHANNEL_WEIGHTS = [
    ("card_present", 0.65),
    ("online", 0.35),
]
FRAUD_TYPE_WEIGHTS = [
    ("account_takeover", 0.25),
    ("card_not_present", 0.25),
    ("counterfeit", 0.20),
    ("friendly_fraud", 0.15),
    ("identity_theft", 0.15),
]


def weighted_choice(rand_col, weighted_labels):
    expr = None
    cumulative = 0.0
    for label, weight in weighted_labels[:-1]:
        cumulative += weight
        expr = F.when(rand_col < F.lit(cumulative), F.lit(label)) if expr is None else expr.when(rand_col < F.lit(cumulative), F.lit(label))
    return expr.otherwise(F.lit(weighted_labels[-1][0]))


def region_to_country(region_col):
    return (
        F.when(region_col.isin("US-East", "US-West"), F.lit("US"))
         .when(region_col == F.lit("EU"), F.lit("UK"))
         .when(region_col == F.lit("APAC"), F.lit("JP"))
         .otherwise(F.lit("BR"))
    )


@pandas_udf(DoubleType())
def faker_spend_bias(id_series: pd.Series) -> pd.Series:
    from faker import Faker
    values = []
    for raw_id in id_series.fillna(0).astype(int).tolist():
        fake = Faker()
        fake.seed_instance(SEED + int(raw_id))
        values.append(fake.random_int(min=94, max=112) / 100.0)
    return pd.Series(values)


@pandas_udf(StringType())
def faker_device_token(key_series: pd.Series) -> pd.Series:
    from faker import Faker
    values = []
    for raw_key in key_series.fillna(0).astype(int).tolist():
        fake = Faker()
        fake.seed_instance(SEED + int(raw_key))
        values.append(fake.lexify(text="?????").upper())
    return pd.Series(values)


def inject_dq_errors(df, seed_offset, null_cols=None):
    """Inject ~15% data quality errors: duplicates + nulls in specified columns."""
    if null_cols is None:
        null_cols = []
    total = df.count()
    n_errors = int(total * DQ_ERROR_RATE)
    n_dupes = int(n_errors * DQ_DUPLICATE_SHARE)
    n_null_per_col = int(n_errors * (1 - DQ_DUPLICATE_SHARE) / max(len(null_cols), 1))

    # Duplicates: sample rows and union them back
    dupe_fraction = min(n_dupes / max(total, 1), 1.0)
    dupes = df.sample(withReplacement=False, fraction=dupe_fraction, seed=SEED + seed_offset)

    # Null injection: for each target column, sample rows and set column to null
    result = df.unionByName(dupes)
    for i, col_name in enumerate(null_cols):
        null_fraction = min(n_null_per_col / max(total, 1), 1.0)
        null_flag = F.rand(SEED + seed_offset + 100 + i) < F.lit(null_fraction)
        result = result.withColumn(col_name, F.when(null_flag, F.lit(None).cast(result.schema[col_name].dataType)).otherwise(F.col(col_name)))

    return result


print(
    {
        "catalog": CATALOG,
        "schema": SCHEMA,
        "volume": VOLUME,
        "volume_root": VOLUME_ROOT,
        "rows": {
            "customers": N_CUSTOMERS,
            "merchants": N_MERCHANTS,
            "transactions": N_TRANSACTIONS,
            "prior_fraud_cases": N_CASES,
        },
        "dq_error_rate": DQ_ERROR_RATE,
        "seed": SEED,
        "output_format": "JSON",
    }
)

In [0]:
import subprocess, os

for subdir in [CUSTOMERS_PATH, MERCHANTS_PATH, TRANSACTIONS_PATH, CASES_PATH]:
    os.makedirs(subdir, exist_ok=True)
    print(f"Directory ready: {subdir}")

print({"volume_root": VOLUME_ROOT, "subdirectories": ["customers", "merchants", "transactions", "prior_fraud_cases"]})

In [0]:
customer_df = (
    spark.range(0, N_CUSTOMERS, 1, DIM_PARTITIONS)
    .withColumn("customer_id", F.format_string("CUST-%05d", F.col("id") + F.lit(1)))
    .withColumn("region_roll", F.rand(SEED + 1))
    .withColumn("risk_roll", F.rand(SEED + 2))
    .withColumn("tenure_roll", F.rand(SEED + 3))
    .withColumn("new_tenure_roll", F.rand(SEED + 4))
    .withColumn("core_tenure_roll", F.rand(SEED + 5))
    .withColumn("mature_tenure_roll", F.rand(SEED + 6))
    .withColumn("home_region", weighted_choice(F.col("region_roll"), HOME_REGION_WEIGHTS))
    .withColumn("risk_segment", weighted_choice(F.col("risk_roll"), RISK_SEGMENT_WEIGHTS))
    .withColumn(
        "tenure",
        F.when(F.col("tenure_roll") < 0.12, (F.floor(F.col("new_tenure_roll") * 3) + 1).cast("int"))
         .when(F.col("tenure_roll") < 0.87, (F.floor(F.col("core_tenure_roll") * 49) + 12).cast("int"))
         .otherwise((F.floor(F.col("mature_tenure_roll") * 120) + 61).cast("int")),
    )
    .withColumn(
        "base_median_spend",
        F.when(F.col("risk_segment") == "low", F.lit(500.0))
         .when(F.col("risk_segment") == "medium", F.lit(1500.0))
         .otherwise(F.lit(3000.0)),
    )
    .withColumn(
        "spend_sigma",
        F.when(F.col("risk_segment") == "low", F.lit(0.48))
         .when(F.col("risk_segment") == "medium", F.lit(0.52))
         .otherwise(F.lit(0.58)),
    )
    .withColumn("faker_spend_bias", faker_spend_bias(F.col("id")))
    .withColumn(
        "avg_monthly_spend",
        F.round(
            F.greatest(
                F.exp(F.log(F.col("base_median_spend")) + F.col("spend_sigma") * F.randn(SEED + 7))
                * F.col("faker_spend_bias")
                * (F.lit(1.0) + F.least(F.col("tenure"), F.lit(120)) / F.lit(2400.0)),
                F.lit(50.0),
            ),
            2,
        ),
    )
    .select("customer_id", "tenure", "home_region", "risk_segment", "avg_monthly_spend")
)

# Inject DQ errors: duplicates + null customer_ids
customer_dirty = inject_dq_errors(customer_df, seed_offset=50, null_cols=["customer_id"])
customer_dirty.write.mode("overwrite").json(CUSTOMERS_PATH)

customer_written = spark.read.json(CUSTOMERS_PATH)
clean_count = customer_written.filter(F.col("customer_id").isNotNull()).dropDuplicates(["customer_id"]).count()
null_ids = customer_written.filter(F.col("customer_id").isNull()).count()
total = customer_written.count()
dupes = total - customer_written.dropDuplicates().count()
print({"path": CUSTOMERS_PATH, "total_rows": total, "clean_unique": clean_count, "null_customer_id": null_ids, "duplicate_rows": dupes, "dq_error_pct": round((null_ids + dupes) / total * 100, 2)})
display(customer_written.orderBy("customer_id").limit(5))

# Keep clean version as temp view for FK joins in transactions
customer_df.createOrReplaceTempView("clean_customers")

In [0]:
merchant_df = (
    spark.range(0, N_MERCHANTS, 1, DIM_PARTITIONS)
    .withColumn("merchant_id", F.format_string("MERCH-%05d", F.col("id") + F.lit(1)))
    .withColumn("mcc_roll", F.rand(SEED + 11))
    .withColumn("risk_roll", F.rand(SEED + 12))
    .withColumn("country_roll", F.rand(SEED + 13))
    .withColumn("mcc_category", weighted_choice(F.col("mcc_roll"), MCC_WEIGHTS))
    .withColumn("risk_tier", weighted_choice(F.col("risk_roll"), MERCHANT_RISK_WEIGHTS))
    .withColumn("country", weighted_choice(F.col("country_roll"), COUNTRY_WEIGHTS))
    .select("merchant_id", "mcc_category", "risk_tier", "country")
)

# Inject DQ errors: duplicates + null merchant_ids
merchant_dirty = inject_dq_errors(merchant_df, seed_offset=60, null_cols=["merchant_id"])
merchant_dirty.write.mode("overwrite").json(MERCHANTS_PATH)

merchant_written = spark.read.json(MERCHANTS_PATH)
clean_count = merchant_written.filter(F.col("merchant_id").isNotNull()).dropDuplicates(["merchant_id"]).count()
null_ids = merchant_written.filter(F.col("merchant_id").isNull()).count()
total = merchant_written.count()
dupes = total - merchant_written.dropDuplicates().count()
print({"path": MERCHANTS_PATH, "total_rows": total, "clean_unique": clean_count, "null_merchant_id": null_ids, "duplicate_rows": dupes, "dq_error_pct": round((null_ids + dupes) / total * 100, 2)})
display(merchant_written.orderBy("merchant_id").limit(5))

# Keep clean version as temp view for FK joins in transactions
merchant_df.createOrReplaceTempView("clean_merchants")

In [0]:
customer_lookup = (
    spark.table("clean_customers")
    .withColumn("customer_idx", F.regexp_extract("customer_id", "(\\d+)$", 1).cast("int") - F.lit(1))
)
merchant_lookup = (
    spark.table("clean_merchants")
    .withColumn("merchant_idx", F.regexp_extract("merchant_id", "(\\d+)$", 1).cast("int") - F.lit(1))
)

base_transactions = (
    spark.range(0, N_TRANSACTIONS, 1, FACT_PARTITIONS)
    .withColumn("txn_id", F.format_string("TXN-%07d", F.col("id") + F.lit(1)))
    .withColumn("customer_idx", F.pmod(F.xxhash64(F.col("id"), F.lit(SEED)), F.lit(N_CUSTOMERS)).cast("int"))
    .withColumn("merchant_idx", F.pmod(F.xxhash64(F.col("id"), F.lit(SEED + 1)), F.lit(N_MERCHANTS)).cast("int"))
    .withColumn("channel_roll", F.rand(SEED + 14))
    .withColumn("days_roll", F.rand(SEED + 15))
    .withColumn("channel", weighted_choice(F.col("channel_roll"), CHANNEL_WEIGHTS))
    .withColumn("days_back", F.floor(F.pow(F.col("days_roll"), F.lit(1.22)) * F.lit(180)).cast("int"))
    .withColumn("txn_date", F.date_sub(F.current_date(), F.col("days_back")))
    .drop("channel_roll", "days_roll")
)

hour_rand = F.rand(SEED + 16)
minute_rand = F.rand(SEED + 17)
second_rand = F.rand(SEED + 18)
card_present_hour = (
    F.when(hour_rand < 0.08, (F.floor(F.rand(SEED + 101) * 2) + 6).cast("int"))
     .when(hour_rand < 0.28, (F.floor(F.rand(SEED + 102) * 3) + 8).cast("int"))
     .when(hour_rand < 0.55, (F.floor(F.rand(SEED + 103) * 4) + 11).cast("int"))
     .when(hour_rand < 0.82, (F.floor(F.rand(SEED + 104) * 4) + 15).cast("int"))
     .when(hour_rand < 0.97, (F.floor(F.rand(SEED + 105) * 3) + 19).cast("int"))
     .otherwise(F.lit(22))
)
online_hour = (
    F.when(hour_rand < 0.10, (F.floor(F.rand(SEED + 106) * 6)).cast("int"))
     .when(hour_rand < 0.30, (F.floor(F.rand(SEED + 107) * 5) + 6).cast("int"))
     .when(hour_rand < 0.65, (F.floor(F.rand(SEED + 108) * 6) + 11).cast("int"))
     .when(hour_rand < 0.90, (F.floor(F.rand(SEED + 109) * 5) + 17).cast("int"))
     .otherwise((F.floor(F.rand(SEED + 110) * 2) + 22).cast("int"))
)

transaction_df = (
    base_transactions
    .join(customer_lookup, on="customer_idx", how="inner")
    .join(merchant_lookup, on="merchant_idx", how="inner")
    .withColumn(
        "timestamp",
        F.make_timestamp(
            F.year("txn_date"),
            F.month("txn_date"),
            F.dayofmonth("txn_date"),
            F.when(F.col("channel") == "card_present", card_present_hour).otherwise(online_hour),
            F.floor(minute_rand * 60).cast("int"),
            F.floor(second_rand * 60).cast("int"),
        ),
    )
    .withColumn("home_country", region_to_country(F.col("home_region")))
    .withColumn("cross_border", F.col("country") != F.col("home_country"))
    .withColumn(
        "mcc_multiplier",
        F.when(F.col("mcc_category") == "travel", F.lit(3.8))
         .when(F.col("mcc_category") == "retail", F.lit(1.55))
         .when(F.col("mcc_category") == "entertainment", F.lit(1.70))
         .when(F.col("mcc_category") == "dining", F.lit(1.05))
         .when(F.col("mcc_category") == "groceries", F.lit(0.72))
         .otherwise(F.lit(0.88)),
    )
    .withColumn(
        "risk_multiplier",
        F.when(F.col("risk_tier") == "high", F.lit(1.22))
         .when(F.col("risk_tier") == "medium", F.lit(1.08))
         .otherwise(F.lit(1.0)),
    )
    .withColumn("channel_multiplier", F.when(F.col("channel") == "online", F.lit(1.14)).otherwise(F.lit(1.0)))
    .withColumn(
        "base_amount",
        F.exp(F.lit(3.91202300543) + F.lit(0.92) * F.randn(SEED + 19))
        * F.col("mcc_multiplier")
        * F.col("risk_multiplier")
        * F.col("channel_multiplier"),
    )
    .withColumn("seed_high_amount", F.rand(SEED + 20) < 0.006)
    .withColumn("seed_unusual_spend", F.rand(SEED + 21) < 0.010)
    .withColumn(
        "amount",
        F.round(
            F.greatest(
                F.when(F.col("seed_high_amount"), F.greatest(F.col("base_amount"), F.lit(2200.0) + F.abs(F.randn(SEED + 22)) * F.lit(2800.0)))
                 .otherwise(F.col("base_amount")),
                F.when(F.col("seed_unusual_spend"), F.col("avg_monthly_spend") * (F.lit(3.25) + F.rand(SEED + 23) * F.lit(2.35)))
                 .otherwise(F.col("base_amount")),
                F.lit(1.0),
            ),
            2,
        ),
    )
    .withColumn(
        "home_device_slots",
        F.when(F.pmod(F.xxhash64(F.col("customer_id"), F.lit(SEED + 24)), F.lit(100)) < 5, F.lit(1))
         .when(F.pmod(F.xxhash64(F.col("customer_id"), F.lit(SEED + 24)), F.lit(100)) < 15, F.lit(2))
         .when(F.pmod(F.xxhash64(F.col("customer_id"), F.lit(SEED + 24)), F.lit(100)) < 35, F.lit(3))
         .when(F.pmod(F.xxhash64(F.col("customer_id"), F.lit(SEED + 24)), F.lit(100)) < 60, F.lit(4))
         .when(F.pmod(F.xxhash64(F.col("customer_id"), F.lit(SEED + 24)), F.lit(100)) < 85, F.lit(5))
         .otherwise(F.lit(6)),
    )
    .withColumn(
        "device_slot",
        F.when(
            (F.col("channel") == "online") & (F.rand(SEED + 25) < 0.06),
            (F.pmod(F.xxhash64(F.col("txn_id"), F.lit(SEED + 26)), F.lit(4)) + F.lit(7)).cast("int"),
        ).otherwise(
            (F.pmod(F.xxhash64(F.col("txn_id"), F.lit(SEED + 27)), F.col("home_device_slots")) + F.lit(1)).cast("int")
        ),
    )
)

customer_window = Window.partitionBy("customer_id").orderBy("timestamp", "txn_id")
device_window = Window.partitionBy("customer_id", "device_slot")

transaction_df = (
    transaction_df
    .withColumn("customer_txn_seq", F.row_number().over(customer_window))
    .withColumn("first_seen_seq", F.min("customer_txn_seq").over(device_window))
    .withColumn("is_new_device", F.col("customer_txn_seq") == F.col("first_seen_seq"))
    .withColumn("device_numeric", F.col("customer_idx") * F.lit(10) + F.col("device_slot"))
    .withColumn(
        "device_id",
        F.concat(
            F.lit("DEV-"),
            faker_device_token(F.col("device_numeric")),
            F.lit("-"),
            F.lpad(F.col("device_numeric").cast("string"), 6, "0"),
        ),
    )
    .withColumn("signal_high_amount_crossborder", (F.col("amount") > 2000) & F.col("cross_border"))
    .withColumn("signal_new_device_online", F.col("is_new_device") & (F.col("channel") == "online"))
    .withColumn("signal_unusual_spend", F.col("amount") > (F.col("avg_monthly_spend") * 3))
    .withColumn(
        "signal_crossborder_highrisk_combo",
        F.col("cross_border") & (F.col("risk_tier") == "high") & (F.col("risk_segment") == "high"),
    )
    .withColumn(
        "signal_count",
        F.col("signal_high_amount_crossborder").cast("int")
        + F.col("signal_new_device_online").cast("int")
        + F.col("signal_unusual_spend").cast("int")
        + F.col("signal_crossborder_highrisk_combo").cast("int"),
    )
    .withColumn(
        "signal_base_prob",
        F.greatest(
            F.when(F.col("signal_high_amount_crossborder"), F.lit(0.25)).otherwise(F.lit(0.0)),
            F.when(F.col("signal_new_device_online"), F.lit(0.15)).otherwise(F.lit(0.0)),
            F.when(F.col("signal_unusual_spend"), F.lit(0.20)).otherwise(F.lit(0.0)),
            F.when(F.col("signal_crossborder_highrisk_combo"), F.lit(0.30)).otherwise(F.lit(0.0)),
        ),
    )
    .withColumn(
        "fraud_prob",
        F.least(
            F.lit(0.005)
            + F.col("signal_base_prob")
            + F.when(F.col("signal_count") > 1, (F.col("signal_count") - 1) * F.lit(0.02)).otherwise(F.lit(0.0))
            + F.when((F.col("channel") == "online") & (F.col("signal_count") > 0), F.lit(0.005)).otherwise(F.lit(0.0)),
            F.lit(0.34),
        ),
    )
    .withColumn("is_fraud", F.rand(SEED + 28) < F.col("fraud_prob"))
    .withColumn(
        "geo_distance_from_home",
        F.round(
            F.when(
                F.col("cross_border"),
                F.greatest(F.exp(F.lit(2.35) + F.lit(0.55) * F.randn(SEED + 29)), F.lit(450.0) + F.abs(F.randn(SEED + 30)) * F.lit(320.0)),
            )
             .when(
                F.col("is_fraud") & F.col("signal_new_device_online"),
                F.greatest(F.exp(F.lit(3.15) + F.lit(0.55) * F.randn(SEED + 31)), F.lit(120.0) + F.abs(F.randn(SEED + 32)) * F.lit(140.0)),
             )
             .otherwise(F.exp(F.lit(2.25) + F.lit(0.62) * F.randn(SEED + 33))),
            2,
        ),
    )
    .select(
        "txn_id",
        "customer_id",
        "merchant_id",
        "amount",
        "timestamp",
        "channel",
        "device_id",
        "geo_distance_from_home",
        "is_fraud",
    )
)

# Inject DQ errors: duplicates + null txn_ids + null customer_ids
txn_dirty = inject_dq_errors(transaction_df, seed_offset=70, null_cols=["txn_id", "customer_id"])
txn_dirty.write.mode("overwrite").json(TRANSACTIONS_PATH)

txn_written = spark.read.json(TRANSACTIONS_PATH)
total = txn_written.count()
null_txn = txn_written.filter(F.col("txn_id").isNull()).count()
null_cust = txn_written.filter(F.col("customer_id").isNull()).count()
dupes = total - txn_written.dropDuplicates().count()
fraud_rate = txn_written.agg(F.round(F.avg(F.col("is_fraud").cast("double")) * 100, 2).alias("fr")).first()["fr"]
print({"path": TRANSACTIONS_PATH, "total_rows": total, "null_txn_id": null_txn, "null_customer_id": null_cust, "duplicate_rows": dupes, "dq_error_pct": round((null_txn + null_cust + dupes) / total * 100, 2), "fraud_rate_pct": fraud_rate})
display(txn_written.orderBy("timestamp").limit(5))
display(txn_written.groupBy("channel", "is_fraud").count().orderBy("channel", "is_fraud"))

# Keep clean version for prior fraud cases
transaction_df.createOrReplaceTempView("clean_transactions")

In [0]:
transactions_for_cases = spark.table("clean_transactions")
fraud_pool = transactions_for_cases.filter(F.col("is_fraud")).withColumn("pool_rank", F.row_number().over(Window.orderBy("txn_id")))
nonfraud_pool = transactions_for_cases.filter(~F.col("is_fraud")).withColumn("pool_rank", F.row_number().over(Window.orderBy("txn_id")))

fraud_count = fraud_pool.count()
nonfraud_count = nonfraud_pool.count()
if fraud_count == 0:
    raise ValueError("Fraud pool is empty; adjust fraud seeding before generating prior fraud cases.")
if nonfraud_count < 20_000:
    raise ValueError("Not enough non-fraud transactions to build the requested prior case sample.")

fraud_case_source = (
    spark.range(0, 30_000, 1, DIM_PARTITIONS)
    .withColumn("pool_rank", (F.pmod(F.col("id"), F.lit(fraud_count)) + F.lit(1)).cast("int"))
    .join(fraud_pool.select("pool_rank", "txn_id", "customer_id", "amount", "timestamp"), on="pool_rank", how="inner")
    .withColumn("source_type", F.lit("fraud_pool"))
    .drop("pool_rank")
)
nonfraud_case_source = (
    spark.range(0, 20_000, 1, DIM_PARTITIONS)
    .withColumn("pool_rank", (F.col("id") + F.lit(1)).cast("int"))
    .join(nonfraud_pool.select("pool_rank", "txn_id", "customer_id", "amount", "timestamp"), on="pool_rank", how="inner")
    .withColumn("source_type", F.lit("nonfraud_pool"))
    .drop("pool_rank")
)

case_source = fraud_case_source.unionByName(nonfraud_case_source)
case_window = Window.orderBy("source_type", "txn_id", "customer_id")

case_df = (
    case_source
    .withColumn("case_seq", F.row_number().over(case_window))
    .withColumn("case_id", F.format_string("CASE-%07d", F.col("case_seq")))
    .withColumn(
        "investigation_outcome",
        F.when(
            F.col("source_type") == "fraud_pool",
            F.when(F.rand(SEED + 41) < 0.90, F.lit("confirmed_fraud"))
             .when(F.rand(SEED + 41) < 0.95, F.lit("false_positive"))
             .otherwise(F.lit("suspicious_not_confirmed")),
        ).otherwise(
            F.when(F.rand(SEED + 42) < 0.15, F.lit("confirmed_fraud"))
             .when(F.rand(SEED + 42) < 0.70, F.lit("false_positive"))
             .otherwise(F.lit("suspicious_not_confirmed")),
        ),
    )
    .withColumn(
        "fraud_type",
        F.when(F.col("investigation_outcome") == "confirmed_fraud", weighted_choice(F.rand(SEED + 43), FRAUD_TYPE_WEIGHTS))
         .otherwise(F.lit(None).cast("string")),
    )
    .withColumn("resolution_days", (F.floor(F.rand(SEED + 44) * 30) + 1).cast("int"))
    .withColumn(
        "resolution_date",
        F.least(
            F.current_timestamp(),
            F.from_unixtime(
                F.unix_timestamp("timestamp")
                + F.col("resolution_days") * F.lit(86400)
                + F.pmod(F.xxhash64(F.col("case_id"), F.lit(SEED + 45)), F.lit(86400))
            ).cast("timestamp"),
        ),
    )
    .withColumn(
        "recovery_amount",
        F.round(
            F.when(
                F.col("investigation_outcome") == "confirmed_fraud",
                F.col("amount") * (F.lit(0.25) + F.rand(SEED + 46) * F.lit(0.70)),
            )
             .when(
                F.col("investigation_outcome") == "suspicious_not_confirmed",
                F.col("amount") * (F.rand(SEED + 47) * F.lit(0.15)),
             )
             .otherwise(F.lit(0.0)),
            2,
        ),
    )
    .select(
        "case_id",
        "txn_id",
        "customer_id",
        "investigation_outcome",
        "resolution_date",
        "fraud_type",
        "recovery_amount",
    )
)

# Inject DQ errors: duplicates + null case_ids + null customer_ids
case_dirty = inject_dq_errors(case_df, seed_offset=80, null_cols=["case_id", "customer_id"])
case_dirty.write.mode("overwrite").json(CASES_PATH)

case_written = spark.read.json(CASES_PATH)
total = case_written.count()
null_case = case_written.filter(F.col("case_id").isNull()).count()
null_cust = case_written.filter(F.col("customer_id").isNull()).count()
dupes = total - case_written.dropDuplicates().count()
print({"path": CASES_PATH, "total_rows": total, "null_case_id": null_case, "null_customer_id": null_cust, "duplicate_rows": dupes, "dq_error_pct": round((null_case + null_cust + dupes) / total * 100, 2)})
display(case_written.orderBy("case_id").limit(5))
display(case_written.groupBy("investigation_outcome").count().orderBy("investigation_outcome"))

In [0]:
# Read back all JSON data as written (with DQ errors)
cust_val = spark.read.json(CUSTOMERS_PATH)
merch_val = spark.read.json(MERCHANTS_PATH)
txn_val = spark.read.json(TRANSACTIONS_PATH)
case_val = spark.read.json(CASES_PATH)

raw_counts = {
    "customers": cust_val.count(),
    "merchants": merch_val.count(),
    "transactions": txn_val.count(),
    "prior_fraud_cases": case_val.count(),
}
print({"raw_row_counts_with_errors": raw_counts})

# DQ summary per entity
dq_summary = []
for name, df, id_col in [
    ("customers", cust_val, "customer_id"),
    ("merchants", merch_val, "merchant_id"),
    ("transactions", txn_val, "txn_id"),
    ("prior_fraud_cases", case_val, "case_id"),
]:
    total = df.count()
    null_id = df.filter(F.col(id_col).isNull()).count()
    deduped = df.dropDuplicates().count()
    dupes = total - deduped
    null_cust = df.filter(F.col("customer_id").isNull()).count() if "customer_id" in df.columns and id_col != "customer_id" else 0
    dq_summary.append((name, total, null_id, null_cust, dupes, round((null_id + null_cust + dupes) / max(total, 1) * 100, 2)))

dq_df = spark.createDataFrame(dq_summary, ["entity", "total_rows", "null_primary_id", "null_customer_id", "duplicate_rows", "dq_error_pct"])
print("\n=== Data Quality Error Summary ===")
display(dq_df)

# File listing
import os
for label, path in [("customers", CUSTOMERS_PATH), ("merchants", MERCHANTS_PATH), ("transactions", TRANSACTIONS_PATH), ("prior_fraud_cases", CASES_PATH)]:
    files = [f for f in os.listdir(path) if f.endswith(".json")]
    total_size_mb = round(sum(os.path.getsize(os.path.join(path, f)) for f in files) / 1_048_576, 2)
    print({"entity": label, "json_files": len(files), "total_size_mb": total_size_mb})

## Generation Summary

This notebook generated JSON files in four subdirectories of `workshop_guy_catalog.fe_bar_a0.staging`:
- `customers/` — 10K base rows + ~15% DQ errors (duplicates, null customer_ids)
- `merchants/` — 5K base rows + ~15% DQ errors (duplicates, null merchant_ids)
- `transactions/` — 500K base rows + ~15% DQ errors (duplicates, null txn_ids, null customer_ids)
- `prior_fraud_cases/` — 50K base rows + ~15% DQ errors (duplicates, null case_ids, null customer_ids)

Fraud patterns are seeded in transactions: high-amount cross-border (~28%), new-device online (~16%), unusual spend (~25%), cross-border high-risk combo (~32%), normal (~0.5%).

The data quality errors are intentional — they simulate realistic raw-landing data for downstream cleansing and data quality pipeline development.